# Module 4 - Topic 6: Building Your First Model
## Live Demo Notebook

**Scenario:** You're a data scientist at Konga Nigeria.
The inventory team asks: *"Can you predict next month's sales so we know how much to stock?"*

This is a **regression problem** - we predict a number (units sold).

We apply the complete Week 5 workflow:
1. Define the problem
2. Load and encode data
3. Split
4. Train (Random Forest)
5. Evaluate (MAE, R²)
6. Feature importance
7. Predict on new scenarios
8. Diagnose fit quality

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

print('Libraries ready.')

---
## Step 1 - Define the Problem

- **Type:** Regression (target is a number: units sold)
- **Features:** product_category, state, month, units_sold_last_month, promotion_active
- **Target:** units_sold_this_month
- **Success metric:** MAE < 15% of average monthly sales

---
## Step 2 - Load and Prepare Data

In [ ]:
df = pd.read_csv('konga_sales_prediction.csv')
print('Shape:', df.shape)
print()
df.head(8)

In [ ]:
print('Data types:')
print(df.dtypes)
print()
print('Missing values:', df.isnull().sum().sum())
print()
print('Target (units_sold_this_month) distribution:')
print(df['units_sold_this_month'].describe().round(1))

In [ ]:
# Encode categorical columns using One-Hot Encoding
# product_category and state are text - ML needs numbers

df_encoded = pd.get_dummies(
    df,
    columns=['product_category', 'state'],
    drop_first=True   # drop one category per group to avoid multicollinearity
)

print(f'Columns before encoding: {df.shape[1]}')
print(f'Columns after encoding:  {df_encoded.shape[1]}')
print()
print('All feature columns:')
for col in df_encoded.columns:
    print(f'  {col}')

In [ ]:
# Separate features from target
X = df_encoded.drop('units_sold_this_month', axis=1)
y = df_encoded['units_sold_this_month']

print(f'Features (X): {X.shape[1]} columns, {X.shape[0]} rows')
print(f'Target  (y): {len(y)} values')
print(f'Average monthly sales: {y.mean():.0f} units')

---
## Step 3 - Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training: {len(X_train)} records | Test: {len(X_test)} records')
print(f'Training avg sales: {y_train.mean():.0f} | Test avg sales: {y_test.mean():.0f}')

---
## Step 4 - Train the Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,  # 100 trees
    max_depth=10,      # limit depth to prevent overfitting
    random_state=42,
    n_jobs=-1          # use all CPU cores
)

rf.fit(X_train, y_train)

print('Training complete.')
print(f'100 trees trained on {len(X_train)} records with {X_train.shape[1]} features.')

# Quick training sanity check
train_mae = mean_absolute_error(y_train, rf.predict(X_train))
print(f'Training MAE: {train_mae:.1f} units')

---
## Step 5 - Evaluate

In [ ]:
y_pred = rf.predict(X_test)

test_mae = mean_absolute_error(y_test, y_pred)
test_r2  = r2_score(y_test, y_pred)
avg_sales = y_test.mean()

print('=== Test Set Evaluation ===')
print(f'MAE:  {test_mae:.1f} units')
print(f'R²:   {test_r2:.3f}')
print()
print(f'Average monthly sales (test set): {avg_sales:.0f} units')
print(f'MAE as % of average: {test_mae/avg_sales*100:.1f}%')
print()
if test_mae / avg_sales < 0.15:
    print('SUCCESS: MAE is below 15% of average sales - meets our success metric.')
else:
    print('MAE exceeds 15% target - consider more features or a different algorithm.')

In [ ]:
# Visualise actual vs predicted
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', s=70)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
        color='red', linestyle='--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Units Sold', fontsize=12)
ax.set_ylabel('Predicted Units Sold', fontsize=12)
ax.set_title('Actual vs Predicted Konga Monthly Sales', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 6 - Feature Importance

In [ ]:
# What drives Konga sales predictions?
importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print('Top 10 most important features:')
for feat, imp in importance.head(10).items():
    bar = '#' * int(imp * 100)
    print(f'  {feat:<35} {imp:.3f}  {bar}')

In [ ]:
# Feature importance chart - top 10
top10 = importance.head(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
top10.plot(kind='barh', ax=ax, color='coral', edgecolor='white')
ax.set_title('Top 10 Features Driving Konga Sales Predictions',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

top_feature = importance.index[0]
print(f'Most important feature: {top_feature}')
print('Business insight: past sales are the strongest predictor of future sales.')

---
## Step 7 - Predict on New Scenarios

In [ ]:
# Create new scenarios - same format as training data, then encode
new_scenarios = pd.DataFrame({
    'product_category':      ['Electronics', 'Books',     'Phones & Tablets'],
    'state':                 ['Lagos',        'Kano',      'Abuja'           ],
    'month':                 [12,             3,           7                 ],
    'units_sold_last_month': [200,            30,          150               ],
    'promotion_active':      [1,              0,           1                 ],
})

# Apply the same encoding
new_encoded = pd.get_dummies(new_scenarios, columns=['product_category', 'state'], drop_first=True)

# Align columns to match training set exactly
new_encoded = new_encoded.reindex(columns=X_train.columns, fill_value=0)

predictions = rf.predict(new_encoded)

print('Sales Predictions for New Scenarios:')
print()
for i, (_, row) in enumerate(new_scenarios.iterrows()):
    print(f'Scenario {i+1}:')
    print(f'  {row.product_category} in {row.state} | Month {row.month}')
    print(f'  Last month: {row.units_sold_last_month} units | Promotion: {"Yes" if row.promotion_active else "No"}')
    print(f'  PREDICTED: {predictions[i]:.0f} units next month')
    print(f'  Inventory recommendation: stock at least {int(predictions[i] * 1.1)} units')
    print()

---
## Step 8 - Diagnose Fit Quality (Topic 4 Checklist)

In [ ]:
train_mae = mean_absolute_error(y_train, rf.predict(X_train))
train_r2  = r2_score(y_train, rf.predict(X_train))
test_mae  = mean_absolute_error(y_test, y_pred)
test_r2   = r2_score(y_test, y_pred)

print('=== Fit Diagnostic ===')
print(f'Training: MAE={train_mae:.1f} units, R²={train_r2:.3f}')
print(f'Test:     MAE={test_mae:.1f} units, R²={test_r2:.3f}')
print(f'R² Gap:   {train_r2 - test_r2:.3f}')
print()

if train_r2 < 0.5:
    print('DIAGNOSIS: UNDERFITTING - low training R². Add features or increase n_estimators.')
elif train_r2 - test_r2 > 0.2:
    print('DIAGNOSIS: OVERFITTING - large train-test gap. Reduce max_depth or add data.')
else:
    print('DIAGNOSIS: GOOD FIT - training and test performance are close.')
    print('This model is ready for stakeholder review.')

---
## Summary: The Complete Week 5 Workflow Applied

| Step | What we did |
|---|---|
| 1. Define | Regression problem - predict units_sold_this_month |
| 2. Prepare | EDA, one-hot encode product_category and state |
| 3. Split | 80/20 train/test |
| 4. Train | RandomForestRegressor(100 trees, max_depth=10) |
| 5. Evaluate | MAE and R² on test set |
| 6. Interpret | Feature importance - past sales dominate |
| 7. Predict | Inventory recommendations for 3 scenarios |
| 8. Diagnose | Applied train/test gap check |

**Week 5 complete.**

**Next - Week 6:** Deep dive into specific algorithms - Linear Regression, Logistic Regression, Decision Trees, Random Forests - with full evaluation metrics.